# gramlab · core-v1 scoreboard

Runs the fixed benchmark of the stable core on Kaggle CPU (no GPU): role linker (learned vs rules vs gold), the ordered
meaning key, entity typing prior, WordNet coverage of layer 0, affect lexicon (if a VAD file is attached as a dataset),
and the index. Training text: AMALGUM (191k silver-parsed sentences, sparse-cloned); evaluation: gold GUM test.
Expected wall time ≈ 10 minutes. Outputs: `results/scoreboard.json`, `results/linker.pkl`.

See `gramlab/core/README.md` for the contracts and `RESULTS.md` run 9 for the baseline numbers.

In [ ]:
import os, sys, glob, subprocess, pathlib, json
WORK = pathlib.Path('/kaggle/working'); os.chdir(WORK)
def sh(c):
    print('$', c); r = subprocess.run(c, shell=True, text=True, capture_output=True)
    if r.returncode: print(r.stdout[-1500:], r.stderr[-1500:])
    return r.returncode == 0
sh('git clone --depth 1 https://github.com/furevych/gramlab.git gramlab')
sys.path.insert(0, str(WORK / 'gramlab'))
GUM = WORK / 'data' / 'gum'; GUM.mkdir(parents=True, exist_ok=True)
for split in ('train', 'dev', 'test'):
    sh(f'curl -sSL -o {GUM}/en_gum-ud-{split}.conllu https://raw.githubusercontent.com/UniversalDependencies/UD_English-GUM/master/en_gum-ud-{split}.conllu')
AMALGUM = 'data/amalgum/amalgum/*/dep/*.conllu'
if not glob.glob(AMALGUM):
    sh('GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/gucorpling/amalgum.git data/amalgum')
print('GUM', sorted(p.name for p in GUM.glob('*.conllu')), '| AMALGUM files', len(glob.glob(AMALGUM)))
# layer-0 preset: WordNet 3.0 files, read by gramlab.core.wordnet (no nltk needed)
if not pathlib.Path('data/wordnet/index.verb').exists():
    sh('curl -sSL -o data/wordnet.zip https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/wordnet.zip')
    import zipfile; zipfile.ZipFile('data/wordnet.zip').extractall('data')      # -> data/wordnet/
os.environ['WORDNET_DIR'] = str(WORK / 'data' / 'wordnet')
from gramlab.core.wordnet import WordNet; print('WordNet', WordNet().stats())
if not pathlib.Path('data/semcor/brown2').exists():
    sh('curl -sSL -o data/semcor.zip https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/semcor.zip')
    zipfile.ZipFile('data/semcor.zip').extractall('data')                        # -> data/semcor/
SEMCOR = str(WORK / 'data' / 'semcor')
# affect lexicon: attach a Kaggle dataset containing NRC-VAD-Lexicon.txt (word, valence, arousal, dominance) to enable the affect row
VAD = next(iter(glob.glob('/kaggle/input/**/NRC-VAD-Lexicon*.txt', recursive=True) + glob.glob('/kaggle/input/**/*vad*.txt', recursive=True)), None)
print('VAD lexicon:', VAD)

In [ ]:
from gramlab.core import bench
board = bench.run(str(GUM), AMALGUM, VAD, out_dir=str(WORK / 'results'), semcor=SEMCOR)


In [ ]:
# a few events through the pipeline, decoded
from gramlab.core import Pipeline, LearnedLinker, read_conllu
pipe = Pipeline(linker=LearnedLinker.load(str(WORK / 'results' / 'linker.pkl')))
test = read_conllu(str(GUM / 'en_gum-ud-test.conllu'))
shown = 0
for s in test:
    for e in pipe.events(s):
        if e.actor and e.patient and e.voice == 'passive':
            print(s.text()[:120]); print('   ', pipe.describe(e)); shown += 1
            break
    if shown >= 6: break
print(pipe.components())

## Reading the scoreboard
- **links**: the linker's F1 against UD gold; passive F1 and role swaps are the numbers that matter for the key.
- **keys**: `key_exact` = the class triple built from predicted roles equals the gold triple; `position_only_control`
  is what the toy map's rule gets on real text (near 0.7 on actives, ~0.08 on passives).
- **typing**: the prior every entity typer has to beat.
- **general**: with WordNet, the share of actions / arguments that got a concept id and how many families the verbs collapse into at depth 2.
- A component change is accepted when its own row does not drop and nothing else moves.